In [1]:
import torch,os
from peft import LoraConfig, get_peft_model, TaskType
from peft import LoraConfig, PeftModel
from trl import SFTTrainer
import pandas as pd

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
print(torch.cuda.is_available())

import re, os, string, random
import numpy as np

/home/dan7hc/.conda/envs/LLM/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True


In [2]:
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
import warnings
warnings.filterwarnings("ignore")

In [3]:
sub_domain = "rest_16_acos" # rest_15, rest_16, rest_16_acos, laptop_16, viabsa_rest
domain = "dataset/" + sub_domain
#domain = "Dataset/vi_res"

domain_name = "restaurant"

path_train = domain + '/csv/train.csv'
df_train = pd.read_csv(path_train,index_col=False)
df_train.fillna("None", inplace=True)

path_dev = domain +  '/csv/dev.csv'
df_dev = pd.read_csv(path_dev,index_col=False)
df_dev.fillna("None", inplace=True)

path_test = domain + '/csv/test.csv'
df_test = pd.read_csv(path_test,index_col=False)
df_test.fillna("None", inplace=True)
df_test.head()

,input,output,paraphrase_format,free_order,json_format,coding_format
0,yum !,"{food quality, NULL, positive, yum}",food quality positive because NULL is yum,[AT] NULL [AC] food quality [SP] positive [OP]...,"[{""category"": ""food quality"", ""aspect"": ""NULL""...","quadruplet_list.append({""category"": ""food qual..."
1,serves really good sushi .,"{food quality, sushi, positive, good}",food quality positive because sushi is good,[AT] sushi [AC] food quality [SP] positive [OP...,"[{""category"": ""food quality"", ""aspect"": ""sushi...","quadruplet_list.append({""category"": ""food qual..."
2,not the biggest portions but adequate .,"{food style_options, portions, neutral, not th...",food style_options neutral because portions is...,[AT] portions [AC] food style_options [SP] neu...,"[{""category"": ""food style_options"", ""aspect"": ...","quadruplet_list.append({""category"": ""food styl..."
3,green tea creme brulee is a must !,"{food quality, green tea creme brulee, positiv...",food quality positive because green tea creme ...,[AT] green tea creme brulee [AC] food quality ...,"[{""category"": ""food quality"", ""aspect"": ""green...","quadruplet_list.append({""category"": ""food qual..."
4,it has great sushi and even better service .,"{food quality, sushi, positive, great};{servic...",food quality positive because sushi is great s...,[AT] sushi [AC] food quality [SP] positive [OP...,"[{""category"": ""food quality"", ""aspect"": ""sushi...","quadruplet_list.append({""category"": ""food qual..."


In [4]:
prompt_type = "coding_format" # coding_format, paraphrase_format, free_order, json_format
model_id = "Salesforce/codet5-base"
learning_rate = 2e-5
num_epochs = 10
batch_size = 16
seed = 42 # 42, 0, 1

"""Set the seed for reproducibility in PyTorch."""
random.seed(seed)            # Python random module.
np.random.seed(seed)         # Numpy module.
torch.manual_seed(seed)      # Sets the seed for generating random numbers.
torch.cuda.manual_seed(seed) # Sets the seed for CUDA.
torch.cuda.manual_seed_all(seed) # Sets the seed for all GPUs.
os.environ['PYTHONHASHSEED'] = str(seed)  # Set PYTHONHASHSEAT to prevent hash-based operations from randomness.

# Configure PyTorch to be deterministic
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
torch.backends.cudnn.deterministic = True  # Avoid nondeterministic algorithms.
torch.backends.cudnn.benchmark = False     # If the input sizes for your neural network do not vary, turning off benchmarking can improve reproducibility.
torch.use_deterministic_algorithms(True)

In [5]:
from transformers import RobertaTokenizer, T5ForConditionalGeneration

cache_dir = "models/"

tokenizer = RobertaTokenizer.from_pretrained(model_id, cache_dir=cache_dir, local_files_only=True)
tokenizer.pad_token = tokenizer.eos_token
model = T5ForConditionalGeneration.from_pretrained(model_id, device_map="auto", trust_remote_code=True,
                                                use_cache=False,
                                                force_download=False,
                                                cache_dir=cache_dir,
                                                local_files_only=True)

# Read training dataset

In [6]:
from prompting_t5 import *

def format_prompt(input_review, prompt_type, domain_name):
    if prompt_type == "coding_format":
        prompt = CODING_FORMAT_PROMPTING

    elif prompt_type == "paraphrase_format":
        prompt = PARAPHRASE_FORMAT_PROMPTING
    
    elif prompt_type == "free_order":
        prompt = FREE_ORDER_FORMAT_PROMPTING
        
    elif prompt_type == "json_format":
        prompt = JSON_FORMAT_PROMPTING
        
    else:
        print("ERROR")
    return prompt.format(input_review=input_review,domain_name=domain_name)

# Create format for input

In [7]:
from datasets import Dataset, DatasetDict

def extract_input_output_for_training(df, type_dataset="train", prompt_type="code_format", domain_name="restaurant"):
    inputs = df["input"].tolist()
    print("Number of samples: ", len(inputs))
    if prompt_type == "coding_format":
        outputs = df["coding_format"].tolist()
    elif prompt_type == "paraphrase_format":
        outputs = df["paraphrase_format"].tolist()
    elif prompt_type == "free_order":
        outputs = df["free_order"].tolist()
    elif prompt_type == "json_format":
        outputs = df["json_format"].tolist()
    else:
        print("error")

    intput_list , output_list, model_input = [], [], []

    for index,review in enumerate(inputs):
        text_prompt = format_prompt(review, prompt_type, domain_name)
        intput_list.append(text_prompt)
        output_list.append(outputs[index])
        model_input.append(text_prompt + "" + outputs[index] + "<end_of_turn>")

    print(len(intput_list),len(output_list), len(model_input))

    print("======> Input: ", intput_list[0])
    print("======> Output: ", output_list[0])
    print("======> Model input: ", model_input[0])

    df_format = pd.DataFrame(list(zip(intput_list, output_list,model_input)), columns =['x_input', 'y_output', 'text'])
    dataset = DatasetDict()
    if type_dataset == "train":
        dataset['train'] = Dataset.from_pandas(df_format)
        return dataset, df_format
    else:
        return intput_list, output_list, model_input

In [8]:
training_dataset,train_df_format = extract_input_output_for_training(df_train, type_dataset="train", prompt_type=prompt_type, domain_name=domain_name)
#======================================
from utils import calculate_max_input_length
max_input_length, max_output_length = calculate_max_input_length(tokenizer,
                                                                 train_df_format["text"].tolist(),
                                                                 train_df_format["y_output"].tolist())
print("max_input_length", max_input_length)
print("max_output_length", max_output_length)

Number of samples:  1530
1530 1530 1530
======> Input:  
def extract_quadruplet_list_from_review(input_review):
    """Extract list of quadruplet sentiments from the input review for the restaurant domain. """

    input_review = "judging from previous posts this used to be a good place , but not any longer ."

    quadruplet_list = []
    for phrase in extract_important_phrase(input_review):
        category = classify_category(phrase)
        aspect = extract_aspect_for_category(phrase, category)  # mentioned in review or implicit aspect
        sentiment = classify_sentiment_for_aspect(phrase, aspect, category)  # positive, negative, or neutral
        opinion = extract_opinion_for_sentiment(phrase, sentiment)  # mentioned in review or implicit opinion
        quadruplet_list.append({
            "category": category,
            "aspect": aspect,
            "sentiment": sentiment,
            "opinion": opinion
        })
    return quadruplet_list

# Run an example
input_review= 

Token indices sequence length is longer than the specified maximum sequence length for this model (525 > 512). Running this sequence through the model will result in indexing errors


max_input_length 829
max_output_length 350


In [9]:
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    processing_class=tokenizer,
    train_dataset=training_dataset["train"],
    args = SFTConfig(
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=4,
        warmup_steps=4,
        num_train_epochs = num_epochs, # Set this for 1 full training run.
        learning_rate = learning_rate,
        neftune_noise_alpha=5,
        bf16 = True,
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        load_best_model_at_end=False,
        seed = seed,
        output_dir = "checkpoint",
        report_to = "none",
        max_seq_length = max_input_length + max_output_length,
        dataset_num_proc = 4,
        logging_steps=50,
        packing = False, # Can make training 5x faster for short sequences.
    ),
)

Truncating train dataset (num_proc=4): 100%|██████████| 1530/1530 [00:00<00:00, 8977.52 examples/s]


In [10]:
import time, os, warnings

start_time= time.time()
trainer.train()
stop_time=time.time()
time_training =stop_time - start_time
print("Training time (seconds): ", time_training)

Step,Training Loss
50,1.074500
100,0.006400
150,0.003500
200,0.002800


Training time (seconds):  380.2516393661499


# Evaluation

In [11]:
x_test,y_test, mode_input_test = extract_input_output_for_training(df_test, type_dataset="test",prompt_type=prompt_type, domain_name=domain_name)

Number of samples:  583
583 583 583
======> Input:  
def extract_quadruplet_list_from_review(input_review):
    """Extract list of quadruplet sentiments from the input review for the restaurant domain. """

    input_review = "yum !"

    quadruplet_list = []
    for phrase in extract_important_phrase(input_review):
        category = classify_category(phrase)
        aspect = extract_aspect_for_category(phrase, category)  # mentioned in review or implicit aspect
        sentiment = classify_sentiment_for_aspect(phrase, aspect, category)  # positive, negative, or neutral
        opinion = extract_opinion_for_sentiment(phrase, sentiment)  # mentioned in review or implicit opinion
        quadruplet_list.append({
            "category": category,
            "aspect": aspect,
            "sentiment": sentiment,
            "opinion": opinion
        })
    return quadruplet_list

# Run an example
input_review= "yum !"
results = extract_quadruplet_list_from_review(input_review)
for quad i

In [18]:
from transformers import GenerationConfig
def evaluate_model_batch(samples,tokenizer,model, max_output_length):
    # Tokenize the batch of samples
    inputs = tokenizer(samples, return_tensors="pt", padding=True, truncation=True).to('cuda')

    # Generation configuration
    generation_config  = GenerationConfig(
            do_sample=False,
            max_new_tokens=max_output_length,
            num_beams=5,
            return_full_text=False,
            early_stopping=True,
            stop_strings=['<end_of_turn>']
        )

    # Generate predictions for the batch
    outputs = model.generate(**inputs, generation_config=generation_config,tokenizer=tokenizer)

    # Decode the outputs
    outputs_decode = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    # Post-process the generated outputs
    processed_outputs = []
    count = 0
    for item in outputs_decode:
        print(item)
        output = item.split("model\n")[1].strip()
        if count == 0:
            #print("item: ", item)
            print("Output222: ", output)
            count+=1
            output = output.replace(")�", ")").replace(")*",")").strip()
        processed_outputs.append(output.strip())
    return processed_outputs

In [19]:
from tqdm import tqdm
y_pred = []
start_time= time.time()
batch_size = 16
for i in tqdm(range(0, len(x_test), batch_size)):
    batch = x_test[i:i+batch_size]  # Create a batch of samples
    preds = evaluate_model_batch(batch, tokenizer, model, max_output_length)
    y_pred.extend(preds)
stop_time=time.time()
inference_time =stop_time - start_time
print("Inference time (seconds): ", inference_time)

  0%|          | 0/37 [00:43<?, ?it/s]

<pad> 
def extract_quadruplet_list_from_review(input_review):
    """Extract list of quadruplet sentiments from the input review for the restaurant domain. """

    input_review = "yum !"

    quadruplet_list = []
    for phrase in extract_important_phrase(input_review):
        category = classify_category(phrase)
        aspect = extract_aspect_for_category(phrase, category)  # mentioned in review or implicit aspect
        sentiment = classify_sentiment_for_aspect(phrase, aspect, category)  # positive, negative, or neutral
        opinion = extract_opinion_for_sentiment(phrase, sentiment)  # mentioned in review or implicit opinion
        quadruplet_list.append({
            "category": category,
            "aspect": aspect,
            "sentiment": sentiment,
            "opinion": opinion
        })
    return quadruplet_list

# Run an example
input_review= "yum !"
results = extract_quadruplet_list_from_review(input_review)
for quad in results:
    print(f"quadruplet_list.append(

IndexError: list index out of range

In [ ]:
print(y_pred[0])
print(y_test[0])

In [ ]:
from utils import *

list_y_test = convert_predict_to_json_format(y_test,prompt_type)
list_y_pred = convert_predict_to_json_format(y_pred,prompt_type)

print(len(list_y_test), len(list_y_pred))

In [ ]:
index = 5
list_y_test[index]

In [ ]:
list_y_pred[index]

In [ ]:
from eval_utils import *
scores, all_labels, all_preds = compute_scores(list_y_pred,list_y_test)
print("Results:")
print(scores)